In [ ]:
Q1. OLTP vs OLAP Architecture 
The company currently stores transactional and analytical workloads in the same SQL  database. 
Tasks
1. Explain why this creates performance issues. 
Ans: When a database tries to handle both operational traffic and heavy queries simultaneously, it faces three main issues:

a) Resource Contention: Operational tasks (e.g., placing an order, updating a user's address) require CPU, memory, and disk I/O to run in milliseconds. Analytical tasks (e.g., calculating monthly revenue across 10 million rows) consume massive amounts of CPU and RAM. When an analytical query runs, it starves operational transactions of resources, slowing down the main app or causing timeouts.

b) Database Locking & Blocking: To keep data consistent during updates or writes, SQL databases lock rows, pages, or entire tables. If a long-running analytical query locks or reads large chunks of a table, simple insert/update transactions have to wait in line, leading to severe application lag.

c) Row-based Storage vs. Analytical Access Patterns: Standard SQL databases store data row-by-row on disk. This is great for looking up single records, but reading a whole column across millions of rows requires reading every single row into memory, wasting disk I/O.

2. OLTP vs. OLAP Workload Comparison 
Area	                         ETL	                            ELT
Transformation point	         Before final load	                    After raw load
Raw copy in destination	         Not always	                            Common
Processing location	             Separate engine or process	            Destination platform
Strong fit	                     Pre-load control and masking	        Modern analytical platforms with scalable compute
Reprocessing	                 May require retained source copies	    Often easier from retained raw data

 
3. Suggest an improved architecture.  
Ans: The standard data engineering solution is to separate the transactional engine from the analytical engine using an ETL/ELT pipeline.
+------------------+         +----------------+         +------------------+         +------------------+
|                  |   ETL   |                |         |                  |         |                  |
|  OLTP Database   |------>  |   Data Lake    |------>  |  Data Warehouse  |------>  |   BI Dashboard   |
| (PostgreSQL/etc.)|         |  (S3/Blob/GCS) |         | (Snowflake/etc.) |         |  (PowerBI/Metabase)|
+------------------+         +----------------+         +------------------+         +------------------+

a) Transactional Layer: Users interact with the app. Data writes to the transactional database without lag.

b) Ingestion / ETL: Change Data Capture (CDC) or scheduled batch jobs extract changes without putting heavy read loads on the production database.

c) Storage & Analytics Layer: Raw data lands in a Data Lake for long-term storage, while structured analytics-ready data is loaded into a Data Warehouse.

d) Presentation Layer: BI dashboards and data analysts query the warehouse directly. Production users and backend services are completely isolated from reporting traffic.
    
4. Explain where:  
-PostgreSQL /SQLServer/MySql
 Ans:   Serving live application traffic, handling user authentications, e-commerce checkouts, inventory tracking, and enforcing precise ACID transactional rules.
-Data warehouse  
 Ans:   Storing large volumes of data in its raw, unstructured, semi-structured, or structured formats (CSV, Parquet, JSON, images, log files) before knowing exactly how it will be queried.
-Data lake 
 Ans:   Running business intelligence dashboards, complex financial reporting, multi-year trend analyses, and heavy aggregation queries across multi-terabyte datasets.
should be used.  


In [ ]:
Q2. ACID vs BASE in Distributed Systems 
QuickCart wants to store clickstream events from millions of users. Tasks 
1. Explain why BASE systems are preferred for clickstream systems. 
Ans: BASE (Basically Available, Soft state, Eventual consistency) systems are preferred here because:

a) Extreme Write Throughput & Scalability: Clickstream events generate thousands of writes per second. BASE systems (like Cassandra, DynamoDB, or Kafka) scale horizontally across many servers without waiting for distributed consensus on every write.

b) High Availability over Instant Consistency: If a server node goes down, a BASE system prioritizes remaining operational (Basically Available) rather than rejecting user events to guarantee instant global agreement.

c) Tolerant to Out-of-Order or Delayed Writes: If a user click event takes 2 seconds to replicate to all nodes (Soft State / Eventual Consistency), it does not impact the user experience or business operations. Losing or slightly delaying 1 out of 100,000 click events is acceptable; bringing down the tracking service is not.

2. Which workloads require ACID compliance?  
Ans: ACID (Atomicity, Consistency, Isolation, Durability) compliance is strictly required wherever data accuracy and transaction integrity are non-negotiable, and partial updates cause financial or operational damage:

a) Financial & Payment Processing: Charging a credit card and updating account balances (e.g., deducting $50 must happen atomically with crediting the seller).

b) Order Placement & Inventory Allocation: Purchasing the last item in stock (prevents double-selling to two simultaneous buyers).

c) User Authentication & Identity: Updating account passwords or permission roles (changes must be immediately visible to block unauthorized access).

Ledger & Accounting Systems: Double-entry bookkeeping where debits and credits must perfectly match.
    
3. Give examples where eventual consistency is acceptable.  
Ans: Eventual consistency works well in scenarios where temporary discrepancies do not harm the core business outcome:

a) Clickstream & User Behavior Analytics: Aggregating total page views for a product page (showing 10,450 vs. 10,452 views for a few seconds does not matter).

b) Social Media Feeds & Likes: Seeing a friend's post or like count update a few seconds late on your feed.

c) Product Recommendations: Updating "Customers who bought this also bought..." lists based on recent browse history.

d) Product Reviews & Star Ratings: A newly posted review appearing after a short replication delay across edge locations.

e) Non-Critical User Profiles: Updating a profile avatar or bio description.


In [ ]:
Q3. Storage Formats & Querying 
Tasks 
1. Compare:  
Ans: 
Format	       Storage Layout	  Type	            Schema Enforcement	               Best Use Case
CSV	           Row-based	      Plain Text	    No (Untyped)	                   Simple data exchange, quick human inspection, small files
JSON	       Row-based	      Plain Text	    Optional / Schema-on-Read	       Web API payloads, semi-structured data, dynamic nested attributes
Avro	       Row-based	      Binary	        Strictly Enforced (JSON Schema)	   Streaming pipelines (Kafka), row-level updates, schema evolution
Parquet	       Columnar	          Binary	        Strictly Enforced (Embedded)	   Analytical queries (OLAP), Data Lakes, Data Warehouses

2. Why do analytical systems prefer columnar storage?  
Ans: In row-based storage, data is written on disk as full rows (Row1_ColA, Row1_ColB, Row1_ColC | Row2_ColA, Row2_ColB, Row2_ColC). 
    In columnar storage (like Parquet), data is grouped and written by columns (ColA_Row1, ColA_Row2 | ColB_Row1, ColB_Row2).

Analytical systems prefer columnar storage because:

I/O Pruning (Projection Pushdown): An analytical query like SELECT SUM(total_amount) FROM sales only needs one column out of potentially 100 columns. 
Columnar storage reads only the total_amount byte range on disk, skipping 99% of unneeded data. 
Row-based storage must scan every single byte of every row into memory.

Vectorized Query Execution: Modern CPUs can apply operations (like additions or filters) across contiguous array-like column data 
in a single CPU instruction cycle (SIMD - Single Instruction, Multiple Data).

3. Explain avro vs json format. 
Ans: 
Apache Avro is a row-oriented binary format with explicit schema support.

It is often useful for:
event exchange;
streaming pipelines;
serialising complete records;
schema-aware messaging. 
Because records are stored row-wise, Avro is not as naturally suited to analytical column pruning as Parquet.
Its schema model is useful when producers and consumers need a clear record contract. 

JSON is plain text, which makes it human-readable but bloated with repetitive keys (e.g., {"user_id": 1}, {"user_id": 2}).
JSON does not require a schema; field types must be guessed or parsed on the fly. 
Handling changing JSON payloads requires custom, error-prone application logic. 

Rule of Thumb: Use JSON for external HTTP APIs and debugging. Use Avro for high-throughput streaming event pipelines (like Apache Kafka). 
    
4. Explain compression benefits in Parquet.  
Ans: Parquet groups data by column, every value stored consecutively in a file block shares the same data type 
(e.g., all integers, all dates, or all strings). This homogeneity enables extreme compression ratios:

> Type-Specific Compression Algorithms: Parquet applies specialized encoding techniques prior to applying standard compression algorithms 
(like Snappy, ZSTD, or Gzip):

    Run-Length Encoding (RLE): Shrinks repeated values (e.g., USA, USA, USA, USA becomes USA x4).

    Dictionary Encoding: Replaces long repeating string values with short integer keys.

    Delta Encoding: Stores only the differences between consecutive numbers or timestamps instead of full values.

> Reduced Disk & Cloud Costs: High compression ratios reduce storage footprints on Amazon S3 / ADLS by up to 75–90% compared to raw CSV/JSON.

> Faster Query Execution: Less physical data on disk means significantly fewer disk I/O reads. Since reading from disk/network is usually 
the main bottleneck in big data processing, reading smaller compressed files directly speeds up query execution.


In [ ]:
Q4. Partitioning, Indexing 
Tasks 
1. Design a partitioning strategy for:  
 -orders table  
                -clickstream data 
Ans: Partitioning splits a large table into smaller, physically separate files or directories based on column values, 
allowing query engines to skip unneeded files entirely (Partition Pruning). 
    Orders Table
Primary Partition Key: order_date (by Month or Day, e.g., year=2026/month=10).

Why: The vast majority of business queries, financial reports, and operational dashboards filter orders by date ranges (e.g., "orders in Q3").

Sub-partitioning (Optional): If the data scale is huge (tens of millions of orders/day), sub-partition by region_id or country to optimize location-specific queries without creating millions of tiny files.

Clickstream Data
Primary Partition Key: event_date (by Day, e.g., date=2026-10-29) + event_type (e.g., page_view, add_to_cart).

Why: Clickstream data is high-volume and naturally time-series driven. Partitioning by day makes date-range filtering fast 
and simplifies data lifecycle policies (e.g., dropping raw clickstream partitions older than 90 days).

Warning (Data Engineering Pitfall): Never partition clickstream data by user_id or session_id. This causes the Small File Problem 
(creating millions of tiny files), which overwhelms file systems like S3/HDFS and severely degrades query performance.


2. Suggest indexing strategy for: 
 -customer lookups - B-Tree Index on customer_id (or email).Why: Looking up a single customer by ID or email requires an exact point lookup. 
    A B-Tree index locates the exact row address in logarithmic time $O(\log N)$ instead of scanning millions of records. 
        
 -order filtering - Strategy: Composite Index on (customer_id, order_date) or Filtered/Partial Index on order_status (e.g., WHERE status = 'PENDING').

Why:

> Searching for a specific customer's order history leverages the composite index to jump straight to that customer's records sorted by date.

> For low-cardinality status columns (like PENDING vs COMPLETED), indexing only active/pending orders saves memory and keeps the index tiny 
while speeding up operational queues. 

 -joins - Strategy: Foreign Key B-Tree Indexes on join columns (e.g., order_items.order_id joining to orders.order_id).
Why: When performing relational joins (JOIN orders ON orders.id = order_items.order_id), the database engine uses foreign key indexes 
to quickly locate matching rows in the child table during Nested Loop or Index Scan operations. 

3. Explain when indexes negatively affect performance.  
Ans: While indexes speed up read operations, they carry notable performance trade-offs:

> Heavy Write Overhead (INSERT, UPDATE, DELETE): Every time data is inserted, updated, or deleted, the database must also 
update all associated indexes in real time. Having too many indexes on a write-heavy OLTP table degrades write throughput significantly.

> Storage & Memory Bloat: Indexes take up disk space and must fit into RAM (buffer pool) for maximum performance. Excessive indexes 
starve the database of cache memory that could otherwise be used for query processing.

> Bad Query Planner Choices: On high-cardinality or poorly maintained index statistics, the query optimizer might mistakenly pick an index 
lookup over a sequential scan, resulting in thousands of random I/O operations that perform worse than reading the table sequentially.

> Ineffective on Low-Cardinality Columns: Creating a standard index on a column with few unique values (e.g., gender or boolean_flag) 
adds index overhead without speeding up reads, as the engine will usually ignore the index and run a full table scan anyway.


In [ ]:
Section B - SQL Engineering 
Q1. Intermediate SQL Queries 
Write SQL queries to: 
1. Find top 10 customers by revenue.  
    SELECT 
    c.customer_id,
    c.customer_name,
    SUM(o.total_amount) AS total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_revenue DESC
LIMIT 10;

2. Find month-over-month sales growth. 
    WITH monthly_sales AS (
    SELECT 
        DATE_TRUNC('month', order_date) AS sales_month,
        SUM(total_amount) AS current_month_revenue
    FROM orders
    GROUP BY DATE_TRUNC('month', order_date)
),
sales_with_lag AS (
    SELECT 
        sales_month,
        current_month_revenue,
        LAG(current_month_revenue) OVER (ORDER BY sales_month) AS previous_month_revenue
    FROM monthly_sales
)
SELECT 
    sales_month,
    current_month_revenue,
    previous_month_revenue,
    ROUND(
        ((current_month_revenue - previous_month_revenue) / NULLIF(previous_month_revenue, 0)) * 100, 
        2
    ) AS mom_growth_percentage
FROM sales_with_lag;
3. Find customers who ordered in consecutive months. 
    WITH customer_months AS (
    SELECT DISTINCT
        customer_id,
        DATE_TRUNC('month', order_date) AS order_month
    FROM orders
),
consecutive_check AS (
    SELECT 
        customer_id,
        order_month,
        LAG(order_month) OVER (
            PARTITION BY customer_id 
            ORDER BY order_month
        ) AS prev_order_month
    FROM customer_months
)
SELECT DISTINCT customer_id
FROM consecutive_check
WHERE prev_order_month = order_month - INTERVAL '1 month';
    
4. Find products never ordered.  
    SELECT 
    p.product_id,
    p.product_name
FROM products p
LEFT JOIN order_items oi ON p.product_id = oi.product_id
WHERE oi.product_id IS NULL;

5. Find revenue contribution percentage by category.  
WITH category_revenue AS (
    SELECT 
        cat.category_name,
        SUM(oi.quantity * oi.unit_price) AS category_sales
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN categories cat ON p.category_id = cat.category_id
    GROUP BY cat.category_name
)
SELECT 
    category_name,
    category_sales,
    ROUND(
        (category_sales / SUM(category_sales) OVER ()) * 100, 
        2
    ) AS revenue_contribution_pct
FROM category_revenue
ORDER BY revenue_contribution_pct DESC;

In [3]:
Q2. Advanced SQL  
Tasks 
1. Rank customers based on total revenue.
    
    WITH customer_revenue AS (
    SELECT 
        c.customer_id,
        c.customer_name,
        SUM(o.total_amount) AS total_revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.customer_name
)
SELECT 
    customer_id,
    customer_name,
    total_revenue,
    DENSE_RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank
FROM customer_revenue;

2. Find running total sales by month. 
    
    WITH monthly_sales AS (
    SELECT 
        DATE_TRUNC('month', order_date) AS sales_month,
        SUM(total_amount) AS monthly_revenue
    FROM orders
    GROUP BY DATE_TRUNC('month', order_date)
)
SELECT 
    sales_month,
    monthly_revenue,
    SUM(monthly_revenue) OVER (
        ORDER BY sales_month 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_running_total
FROM monthly_sales 
ORDER BY sales_month;

3. Find the highest selling product per category. 
    
WITH product_sales AS (
    SELECT 
        cat.category_name,
        p.product_id,
        p.product_name,
        SUM(oi.quantity * oi.unit_price) AS total_product_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN categories cat ON p.category_id = cat.category_id
    GROUP BY cat.category_name, p.product_id, p.product_name
),
ranked_products AS (
    SELECT 
        category_name,
        product_id,
        product_name,
        total_product_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY category_name 
            ORDER BY total_product_revenue DESC
        ) AS rank_in_category
    FROM product_sales
)
SELECT 
    category_name,
    product_name,
    total_product_revenue
FROM ranked_products
WHERE rank_in_category = 1;
    
4. Find 7-day rolling average sales.  
    
    WITH daily_sales AS (
    SELECT 
        DATE_TRUNC('day', order_date) AS sale_date,
        SUM(total_amount) AS daily_revenue
    FROM orders
    GROUP BY DATE_TRUNC('day', order_date)
)
SELECT 
    sale_date,
    daily_revenue,
    ROUND(
        AVG(daily_revenue) OVER (
            ORDER BY sale_date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 
        2
    ) AS rolling_7day_avg_revenue
FROM daily_sales
ORDER BY sale_date;


SyntaxError: invalid syntax (2539270130.py, line 1)

In [1]:
Q3. SQL Optimization Scenario 
The following query takes 18 minutes: 
SELECT c.customer_name, 
 SUM(o.quantity * o.unit_price) 
FROM orders o 
JOIN customers c 
ON o.customer_id = c.customer_id
WHERE order_date >= '2025-01-01' 
GROUP BY c.customer_name; 
Tasks 
1. Identify possible bottlenecks.  
Full Table Scan on orders: The database is likely reading every block of the orders table from disk because there is no index on order_date.

Heavy Join on Non-Filtered Column: The query joins the full orders table to customers before applying or aggregating filtered rows, 
spilling unindexed intermediate data to temporary disk storage.

Expensive GROUP BY on a Text Field (customer_name): Grouping by strings requires expensive text comparisons and 
hash table lookups instead of integer comparisons on customer_id.

Missing Foreign Key Index: If orders.customer_id is unindexed, the database engine cannot perform efficient Index Nested Loop 
or Hash Join lookups.
    
2. Suggest indexes.  
-- 1. Composite Index on orders (Filter column first, then Join & Calculation columns)
CREATE INDEX idx_orders_date_cust_calc 
ON orders (order_date, customer_id, quantity, unit_price);

-- 2. Primary Key / B-Tree Index on customers
CREATE INDEX idx_customers_id_name 
ON customers (customer_id, customer_name);

3. Explain execution plan analysis.  
To diagnose the query, run EXPLAIN ANALYZE (or EXPLAIN in MS SQL Server / MySQL):

SQL
EXPLAIN ANALYZE 
SELECT c.customer_name, SUM(o.quantity * o.unit_price) 
FROM orders o JOIN customers c ON o.customer_id = c.customer_id
WHERE order_date >= '2025-01-01' GROUP BY c.customer_name;
Key Execution Plan Indicators to Watch
> Scan Type:

Bad: Seq Scan (Sequential Scan) or Table Scan on orders.

Good: Index Scan or Index Range Scan using idx_orders_date_cust_calc.

> Join Algorithm:

Bad: Nested Loop across millions of rows without indexes.

Good: Hash Join or Merge Join using indexed keys.

WorkMem / Temp DB Spills: Look for External merge Disk or temporary disk writes. This means the GROUP BY or JOIN exceeded RAM limits, 
forcing disk swap operations that severely degrade performance.

Cost & Rows Estimates: Compare estimated rows vs actual rows. A large mismatch indicates stale table statistics 
(resolved by running ANALYZE orders;).


SyntaxError: invalid syntax (3512033026.py, line 1)

In [ ]:
Q4. Database Design 
Design schema for: 
1. Shipment tracking  
2. Product inventory  
3. Customer support tickets  

Schema Overview & Entity-Relationship Diagram:
[ customers ] ───< [ orders ] ───< [ order_items ] >─── [ products ] ───< [ inventory_levels ] >─── [ warehouses ]
      │               │                
      │               └───< [ shipments ] ───< [ shipment_events ]
      │
      └───< [ support_tickets ] ───< [ ticket_messages ]

Requirements: 
• normalized design  
• primary keys  

-- Customers Table
CREATE TABLE customers (
    customer_id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_name VARCHAR(100) NOT NULL,
    email VARCHAR(255) NOT NULL UNIQUE,
    phone VARCHAR(20),
    created_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
);

-- Warehouses Table
CREATE TABLE warehouses (
    warehouse_id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    warehouse_name VARCHAR(100) NOT NULL,
    location_address TEXT NOT NULL,
    country_code VARCHAR(3) NOT NULL
);

-- Products Table
CREATE TABLE products (
    product_id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    sku VARCHAR(50) NOT NULL UNIQUE,
    product_name VARCHAR(150) NOT NULL,
    category VARCHAR(50) NOT NULL,
    unit_price DECIMAL(10, 2) NOT NULL,
    created_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
); 

2. Product Inventory Module
To keep the database in 3NF, inventory counts are separated by warehouse location.
-- Inventory Levels Table
CREATE TABLE inventory_levels (
    inventory_id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    product_id INT NOT NULL REFERENCES products(product_id),
    warehouse_id INT NOT NULL REFERENCES warehouses(warehouse_id),
    quantity_on_hand INT NOT NULL DEFAULT 0 CHECK (quantity_on_hand >= 0),
    quantity_reserved INT NOT NULL DEFAULT 0 CHECK (quantity_reserved >= 0),
    reorder_threshold INT NOT NULL DEFAULT 10,
    last_updated TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT uq_product_warehouse UNIQUE (product_id, warehouse_id)
);
    
• foreign keys  
3. Shipment Tracking Module
-- Orders Table
CREATE TABLE orders (
    order_id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_id INT NOT NULL REFERENCES customers(customer_id),
    order_date TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP,
    order_status VARCHAR(20) NOT NULL CHECK (order_status IN ('PENDING', 'PROCESSING', 'SHIPPED', 'DELIVERED', 'CANCELLED')),
    total_amount DECIMAL(10, 2) NOT NULL
);

-- Order Items Table
CREATE TABLE order_items (
    order_item_id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    order_id INT NOT NULL REFERENCES orders(order_id) ON DELETE CASCADE,
    product_id INT NOT NULL REFERENCES products(product_id),
    quantity INT NOT NULL CHECK (quantity > 0),
    unit_price DECIMAL(10, 2) NOT NULL
);

-- Shipments Table
CREATE TABLE shipments (
    shipment_id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    order_id INT NOT NULL REFERENCES orders(order_id),
    warehouse_id INT NOT NULL REFERENCES warehouses(warehouse_id),
    carrier_name VARCHAR(50) NOT NULL,
    tracking_number VARCHAR(100) NOT NULL UNIQUE,
    shipment_status VARCHAR(30) NOT NULL CHECK (shipment_status IN ('LABEL_CREATED', 'IN_TRANSIT', 'OUT_FOR_DELIVERY', 'DELIVERED', 'FAILED')),
    shipped_at TIMESTAMP WITH TIME ZONE,
    estimated_delivery DATE,
    delivered_at TIMESTAMP WITH TIME ZONE
);

-- Shipment Tracking Events (Audit/History Log)
CREATE TABLE shipment_events (
    event_id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    shipment_id INT NOT NULL REFERENCES shipments(shipment_id) ON DELETE CASCADE,
    location_description VARCHAR(255),
    event_status VARCHAR(50) NOT NULL,
    event_timestamp TIMESTAMP WITH TIME ZONE NOT NULL,
    notes TEXT
);

4. Customer Support Tickets Module
-- Support Tickets Table
CREATE TABLE support_tickets (
    ticket_id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_id INT NOT NULL REFERENCES customers(customer_id),
    order_id INT REFERENCES orders(order_id), -- Optional link to an order
    shipment_id INT REFERENCES shipments(shipment_id), -- Optional link to a shipment
    subject VARCHAR(200) NOT NULL,
    ticket_status VARCHAR(20) NOT NULL DEFAULT 'OPEN' CHECK (ticket_status IN ('OPEN', 'IN_PROGRESS', 'RESOLVED', 'CLOSED')),
    priority VARCHAR(10) NOT NULL DEFAULT 'MEDIUM' CHECK (priority IN ('LOW', 'MEDIUM', 'HIGH', 'CRITICAL')),
    created_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
);

-- Support Ticket Messages / Conversation Threadss
CREATE TABLE ticket_messages (
    message_id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    ticket_id INT NOT NULL REFERENCES support_tickets(ticket_id) ON DELETE CASCADE,
    sender_type VARCHAR(10) NOT NULL CHECK (sender_type IN ('CUSTOMER', 'AGENT', 'SYSTEM')),
    sender_id INT NOT NULL, -- Refers to customer_id or agent_id
    message_body TEXT NOT NULL,
    created_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
);



• indexing recommendations  
1. Foreign Key B-Tree Indexes
By default, standard SQL engines do NOT automatically index foreign keys. Creating B-Tree indexes on foreign keys avoids full table scans 
during joins:

 -- Foreign key lookups
CREATE INDEX idx_inventory_product_id ON inventory_levels(product_id);
CREATE INDEX idx_inventory_warehouse_id ON inventory_levels(warehouse_id);
CREATE INDEX idx_orders_customer_id ON orders(customer_id);
CREATE INDEX idx_shipments_order_id ON shipments(order_id);
CREATE INDEX idx_shipment_events_shipment_id ON shipment_events(shipment_id);
CREATE INDEX idx_support_tickets_customer_id ON support_tickets(customer_id);
CREATE INDEX idx_support_tickets_order_id ON support_tickets(order_id);

2. Operational & Search Point-Lookups 
-- Fast tracking number & SKU search
CREATE UNIQUE INDEX idx_shipments_tracking ON shipments(tracking_number);
CREATE UNIQUE INDEX idx_products_sku ON products(sku);

-- Lookup ticket history for a given shipment/order
CREATE INDEX idx_tickets_shipment_id ON support_tickets(shipment_id) WHERE shipment_id IS NOT NULL; 

3. Partial & Composite Indexes for Queue Filtering 
-- Fast retrieval of active/unresolved support tickets
CREATE INDEX idx_open_tickets ON support_tickets(priority, created_at) 
WHERE ticket_status IN ('OPEN', 'IN_PROGRESS');

-- Fast tracking status timeline lookup
CREATE INDEX idx_shipment_events_timeline ON shipment_events(shipment_id, event_timestamp DESC);

-- Inventory alert queries (identifying items needing reorder)
CREATE INDEX idx_inventory_reorder ON inventory_levels(warehouse_id) 
WHERE (quantity_on_hand - quantity_reserved) <= reorder_threshold;

In [ ]:
Section C - Python + Pandas EngineeringQ1. Data Ingestion Pipeline1. Read All DatasetsThe datasets (clickstream.csv, customers.csv, orders.csv, and products.csv) are ingested into Pandas DataFrames:Pythonimport pandas as pd

# Data Ingestion
clickstream_df = pd.read_csv('clickstream.csv')
customers_df = pd.read_csv('customers.csv')
orders_df = pd.read_csv('orders.csv')
products_df = pd.read_csv('products.csv')

# Dataset summary overview
print(f"Clickstream shape: {clickstream_df.shape}")  # (2000, 6)
print(f"Customers shape:  {customers_df.shape}")   # (100, 5)
print(f"Orders shape:     {orders_df.shape}")      # (500, 7)
print(f"Products shape:   {products_df.shape}")    # (10, 6)
DatasetTotal RowsTotal ColumnsPrimary Key CandidateForeign Keysclickstream.csv2,0006event_idcustomer_idcustomers.csv1005customer_idN/Aorders.csv5007order_idcustomer_id, product_idproducts.csv106product_idsupplier_id2. Schema & Data Consistency ValidationThe schema and data integrity across all four tables reveal the following baseline types and structural properties:Pythondef validate_schema(df, df_name, expected_dtypes):
    print(f"--- Schema Validation for {df_name} ---")
    for col, expected in expected_dtypes.items():
        actual = str(df[col].dtype)
        status = "OK" if actual == expected else f"MISMATCH (Expected: {expected}, Found: {actual})"
        print(f"Column '{col}': {status}")

# Example validation execution
validate_schema(customers_df, "Customers", {'customer_id': 'int64', 'email': 'object'})
Clickstream (clickstream.csv): customer_id is parsed as float64 due to missing values (NaN). event_timestamp contains mixed timestamp formats (ISO-8601, UK standard DD/MM/YYYY HH:MM, standard SQL timestamps YYYY-MM-DD HH:MM:SS).Customers (customers.csv): Clean schema with 100 unique customer IDs (int64). Email syntax standard across entries.Orders (orders.csv): Valid foreign keys referencing customers and products. Unit prices match catalog selling prices perfectly.Products (products.csv): 10 products with consistent cost vs. selling prices ($selling\_price > cost\_price$).3. Detect Duplicate CustomersPython# Check exact duplicates and key constraints on customers.csv
dup_ids = customers_df[customers_df.duplicated(subset=['customer_id'], keep=False)]
dup_emails = customers_df[customers_df.duplicated(subset=['email'], keep=False)]

print(f"Duplicate Customer IDs found: {len(dup_ids)}")      # Result: 0
print(f"Duplicate Emails found: {len(dup_emails)}")         # Result: 0
Findings: In customers.csv, there are 0 duplicate customer IDs and 0 duplicate emails.Clickstream Context: In clickstream.csv, multiple records exist per customer as expected for multi-event browsing activity.4. Identify Invalid RecordsAudit of invalid, incomplete, or corrupted records across datasets:Python# 1. Clickstream invalid records
cs_null_customers = clickstream_df['customer_id'].isna().sum()   # 60 missing customer_ids
cs_null_urls = clickstream_df['page_url'].isna().sum()             # 83 missing page_urls
cs_null_devices = clickstream_df['device_type'].isna().sum()       # 424 missing device_types
dup_events = clickstream_df['event_id'].duplicated().sum()         # 37 duplicate event_id occurrences (73 total rows affected)

# 2. Orders invalid records (future date anomaly)
orders_df['parsed_date'] = pd.to_datetime(orders_df['order_date'], errors='coerce')
future_orders = orders_df[orders_df['parsed_date'] > '2026-11-02'] # 179 future dates (up to 2028-04-14)
Issue TypeDatasetAffected Record CountDescriptionDuplicate Event IDsclickstream.csv73 rows (37 dups)Duplicate event_id values with conflicting user event payloads.Missing Primary Attributesclickstream.csv60 rowsMissing customer_id (anonymous traffic or log drop).Missing Page URLsclickstream.csv83 rowsNull values in page_url.Missing Device Infoclickstream.csv424 rowsNull values in device_type.Unstandardized Eventsclickstream.csv2,000 rowsTypos (clik, serach, log_out) and inconsistent casing (ADD TO CART, pageview).Future Order Datesorders.csv179 rowsorder_date timestamps populated in future years (e.g., 2027–2028).Q2. Data Cleaning & TransformationPipeline ImplementationPythonimport pandas as pd
import numpy as np

def clean_data_pipeline(clickstream, customers, orders, products):
    # --- Task 1: Standardize City Names ---
    customers_clean = customers.copy()
    customers_clean['city'] = customers_clean['city'].astype(str).str.strip().str.title()

    # --- Task 2: Convert Timestamps Correctly ---
    clickstream_clean = clickstream.copy()
    clickstream_clean['event_timestamp'] = pd.to_datetime(
        clickstream_clean['event_timestamp'], format='mixed', utc=True
    )
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'])

    orders_clean = orders.copy()
    orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'])

    # --- Task 3 & 4: Handle Missing Values & Remove Corrupted Rows ---
    # 1. Deduplicate clickstream event_ids
    clickstream_clean = clickstream_clean.drop_duplicates(subset=['event_id'], keep='first')

    # 2. Normalize event_type and device_type
    def normalize_event(val):
        if pd.isna(val): return val
        val_clean = str(val).strip().lower().replace('-', '_').replace(' ', '_')
        mapping = {
            'clik': 'click', 'serach': 'search', 'pageview': 'page_view',
            'log_out': 'logout', 'add_to_cart': 'add_to_cart'
        }
        return mapping.get(val_clean, val_clean)

    clickstream_clean['event_type'] = clickstream_clean['event_type'].apply(normalize_event)
    clickstream_clean['device_type'] = clickstream_clean['device_type'].str.strip().str.lower()

    # 3. Impute missing device_type and page_url for analytics retention
    clickstream_clean['device_type'] = clickstream_clean['device_type'].fillna('unknown')
    clickstream_clean['page_url'] = clickstream_clean['page_url'].fillna('/unknown')

    # 4. Remove corrupted rows missing customer_id for session analytics
    clickstream_clean = clickstream_clean.dropna(subset=['customer_id'])
    clickstream_clean['customer_id'] = clickstream_clean['customer_id'].astype(int)

    # 5. Quarantine corrupted orders (future dates)
    valid_orders = orders_clean[orders_clean['order_date'] <= '2026-11-02'].copy()

    return clickstream_clean, customers_clean, valid_orders, products
Pipeline Transformation MetricsCity Standardization: Applied .str.strip().str.title() across 100 customer records (15 unique European cities verified).Timestamp Normalization: Converted all mixed string formats into ISO UTC datetime64[ns, UTC].Imputation & Corrupted Row Removal:Removed 37 duplicate event IDs, leaving 1,963 clean records.Dropped 58 records missing customer_id, resulting in 1,905 verified customer event rows.Imputed null device_type (414 remaining in deduplicated set) as 'unknown'.Q3. Clickstream Analytics1. Most Visited PagesAnalyzing page visit distribution across all events vs. explicit page_view user events:Python# All event types by URL (Cleaned dataset, N=1,963)
top_pages_all = clickstream_clean['page_url'].value_counts()

# Page_view events specifically
top_page_views = clickstream_clean[clickstream_clean['event_type'] == 'page_view']['page_url'].value_counts()
RankPage URLAll Events CountExplicit page_view Count1/products206332/search200283/product/105200274/home200345/account197386/orders186277/product/101185298/cart175289/checkout1732910/help161342. Session Count CalculationApplying standard web analytics logic (30-minute inactivity boundary per customer):Python# Sessionization logic
cs_sorted = clickstream_clean.dropna(subset=['customer_id']).sort_values(['customer_id', 'event_timestamp'])
cs_sorted['time_diff'] = cs_sorted.groupby('customer_id')['event_timestamp'].diff()
cs_sorted['new_session'] = (cs_sorted['time_diff'] > pd.Timedelta(minutes=30)) | (cs_sorted['time_diff'].isna())
cs_sorted['session_id'] = cs_sorted.groupby('customer_id')['new_session'].cumsum()

total_sessions = cs_sorted.groupby(['customer_id', 'session_id']).ngroups
print(f"Total Customer Sessions (30-min window): {total_sessions}")
Total 30-Minute Inactivity Sessions: 1,901 sessions across 100 active customers.Average Events Per Session: 1.002 events per session (indicating sparse clickstream events per timestamp).3. Mobile vs. Desktop Traffic PercentagePython# Filter known device traffic (Mobile vs Desktop)
known_devices = clickstream_clean[clickstream_clean['device_type'].isin(['mobile', 'desktop'])]
device_counts = known_devices['device_type'].value_counts()

mobile_pct = (device_counts['mobile'] / len(known_devices)) * 100
desktop_pct = (device_counts['desktop'] / len(known_devices)) * 100

print(f"Mobile Share:  {mobile_pct:.2f}%")   # 59.64%
print(f"Desktop Share: {desktop_pct:.2f}%")  # 40.36%
Traffic Share (Mobile vs. Desktop)
==================================
Mobile  [####################        ] 59.64% (656 events)
Desktop [##############              ] 40.36% (444 events)
Traffic Breakdown CategoryEvent Count (Cleaned)Share of Mobile/DesktopShare of Total TrafficMobile65659.64%33.42%Desktop44440.36%22.62%Tablet449—22.87%Unknown / Missing414—21.09%Section D - ETL/ELT + AirflowQ1. ETL vs. ELT Architecture1. Comparison Matrix: ETL vs. ELTFeature / MetricTraditional ETL (Extract, Transform, Load)Modern ELT (Extract, Load, Transform)Transformation TimingTransformed before loading into the target dataset.Loaded raw first, transformed inside cloud warehouse.Compute LocationSpecialized external ETL server (e.g., Informatica, Talend).Native cloud data warehouse compute engines (Snowflake, BigQuery).Data FlexibilityRigid. Modifying business logic requires updating pipeline code.High. Raw data preserved; logic changes via SQL/dbt models.ScalabilityLimited by dedicated ETL server RAM/CPU capacity.Highly scalable MPP (Massively Parallel Processing) execution.LatencyHigher processing time before data is available for reporting.Near real-time loading into staging; asynchronous transformation.2. Why Cloud Warehouses Prefer ELTModern cloud warehouses (Snowflake, BigQuery, Databricks, Redshift) favor ELT for three key technical reasons:Decoupled Compute and Storage: Cloud storage is inexpensive, allowing organizations to store raw payloads indefintely without upfront transformations.Massively Parallel Processing (MPP): Cloud warehouses scale compute clusters elastically on-demand, executing SQL transformations faster than standalone ETL servers.Replayability & Auditability: Preserving raw ingest tables allows data engineers to fix bugs or backfill transformation logic historically without re-extracting data from source systems.3. QuickCart Modern ELT Pipeline Architecture[Sources: QuickCart CSVs / Postgres DB / App Events]
                       │
                       ▼
           1. EXTRACT & INGEST
      (Airflow / Fivetran / Airbyte)
                       │
                       ▼
           2. LOAD RAW STAGING
  (Snowflake / BigQuery: `raw_quickcart` schema)
                       │
                       ▼
         3. TRANSFORM IN-WAREHOUSE
      (dbt models / SQL Transformations)
                       │
       ┌───────────────┴───────────────┐
       ▼                               ▼
 [Analytics Data Marts]     [Data Quality Tests]
  (`analytics` schema)       (dbt test / Great Expectations)
       │
       ▼
 [BI & Reporting] (Looker, Tableau, Metabase)
Q2. Airflow DAG DevelopmentThe Python script below defines an Apache Airflow DAG enforcing daily execution at 2 AM, file presence checks via sensors, staging ingestion, dbt transformations, retries, logging, and failure email callbacks.Pythonfrom datetime import datetime, timedelta
import logging
from airflow import DAG
from airflow.sensors.filesystem import FileSensor
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.operators.email import EmailOperator
import pandas as pd
import psycopg2

# Default DAG configuration
default_args = {
    'owner': 'quickcart_data_eng',
    'depends_on_past': False,
    'email': ['alerts@quickcart.com'],
    'email_on_failure': True,
    'email_on_retry': False,
    'retries': 3,
    'retry_delay': timedelta(minutes=5),
    'on_failure_callback': lambda context: logging.error(
        f"DAG Task Failed: {context.get('task_instance').task_id}"
    )
}

# Define DAG
with DAG(
    dag_id='quickcart_daily_elt_pipeline',
    default_args=default_args,
    description='Daily ELT Ingestion and Analytics Pipeline for QuickCart',
    schedule_interval='0 2 * * *',  # Daily at 2:00 AM
    start_date=datetime(2025, 1, 1),
    catchup=False,
    max_active_runs=1,
    tags=['quickcart', 'elt', 'daily']
) as dag:

    # 1. File Sensors - Check for presence of source CSVs
    sensor_customers = FileSensor(
        task_id='sense_customers_csv',
        filepath='/opt/airflow/dags/data/customers.csv',
        poke_interval=30,
        timeout=600,
        mode='reschedule'
    )

    sensor_orders = FileSensor(
        task_id='sense_orders_csv',
        filepath='/opt/airflow/dags/data/orders.csv',
        poke_interval=30,
        timeout=600,
        mode='reschedule'
    )

    # 2. Extract & Validate Schema Step
    def validate_and_extract_raw():
        logging.info("Starting schema and data validation step...")
        cust = pd.read_csv('/opt/airflow/dags/data/customers.csv')
        orders = pd.read_csv('/opt/airflow/dags/data/orders.csv')

        if cust.empty or orders.empty:
            raise ValueError("Extracted CSV dataset is empty!")

        logging.info(f"Successfully validated {len(cust)} customers and {len(orders)} orders.")

    task_validate = PythonOperator(
        task_id='extract_and_validate',
        python_callable=validate_and_extract_raw
    )

    # 3. Load Raw Data into Warehouse Staging Tables
    def load_to_staging():
        logging.info("Loading validated datasets into staging tables...")
        # Database connection & bulk COPY statement simulation
        logging.info("Raw data loaded into schema `staging`.")

    task_load_staging = PythonOperator(
        task_id='load_raw_to_staging',
        python_callable=load_to_staging
    )

    # 4. Trigger Transformation Step (In-Warehouse dbt Run)
    task_transform = BashOperator(
        task_id='trigger_dbt_transformations',
        bash_command='dbt run --select tag:daily_quickcart'
    )

    # 5. Email Notification on Critical Failure Task
    send_failure_alert = EmailOperator(
        task_id='send_failure_email',
        to='data-alerts@quickcart.com',
        subject='[AIRFLOW FAILURE] QuickCart Daily ELT Pipeline Failed',
        html_content='<h3>ELT Pipeline execution failed. Inspect task logs in Airflow UI.</h3>',
        trigger_rule='one_failed'
    )

    # --- Task Dependencies ---
    [sensor_customers, sensor_orders] >> task_validate >> task_load_staging >> task_transform
    [task_validate, task_load_staging, task_transform] >> send_failure_alert